# Week 1: Data Validation and Exploratory Data Analysis

## Project Objective

This project focuses on multi-touch marketing attribution. The aim is to understand how different marketing touchpoints contribute to customer purchases.

The available datasets include web events, transactions, customers, products, and campaign metadata.

The current phase focuses on:
- Dataset validation
- Customer journey analysis
- Funnel analysis
- First-touch attribution preparation
- Last-touch attribution preparation
- Linear attribution preparation

## Data Limitation

The shared datasets do not currently include ad spend, campaign cost, budget, CPC, or CPM data.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# -----------------------------
# 0. Project paths and settings
# -----------------------------

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
ATTRIBUTION_SAMPLE_SIZE = 5000   # Used because events.csv is large on a local laptop
LOOKBACK_DAYS = 30

np.random.seed(RANDOM_SEED)

In [3]:
# -----------------------------
# 1. Load datasets
# -----------------------------

required_files = {
    "events": DATA_DIR / "events.csv",
    "transactions": DATA_DIR / "transactions.csv",
    "customers": DATA_DIR / "customers.csv",
    "products": DATA_DIR / "products.csv",
    "campaigns": DATA_DIR / "campaigns.csv",
}

missing_files = [name for name, path in required_files.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

events = pd.read_csv(required_files["events"], low_memory=False)
transactions = pd.read_csv(required_files["transactions"], low_memory=False)
customers = pd.read_csv(required_files["customers"], low_memory=False)
products = pd.read_csv(required_files["products"], low_memory=False)
campaigns = pd.read_csv(required_files["campaigns"], low_memory=False)

datasets = {
    "events": events,
    "transactions": transactions,
    "customers": customers,
    "products": products,
    "campaigns": campaigns,
}

print("Data loaded successfully.")
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows and {df.shape[1]} columns")
    

Data loaded successfully.
events: 2,000,000 rows and 12 columns
transactions: 103,127 rows and 9 columns
customers: 100,000 rows and 7 columns
products: 2,000 rows and 6 columns
campaigns: 50 rows and 7 columns


In [4]:
# -----------------------------
# 2. Basic validation
# -----------------------------

expected_columns = {
    "events": [
        "event_id", "timestamp", "customer_id", "session_id", "event_type",
        "product_id", "device_type", "traffic_source", "campaign_id",
        "page_category", "session_duration_sec", "experiment_group"
    ],
    "transactions": [
        "transaction_id", "timestamp", "customer_id", "product_id", "quantity",
        "discount_applied", "gross_revenue", "campaign_id", "refund_flag"
    ],
    "customers": [
        "customer_id", "signup_date", "country", "age", "gender",
        "loyalty_tier", "acquisition_channel"
    ],
    "products": [
        "product_id", "category", "brand", "base_price", "launch_date", "is_premium"
    ],
    "campaigns": [
        "campaign_id", "channel", "objective", "start_date", "end_date",
        "target_segment", "expected_uplift"
    ],
}

for name, expected in expected_columns.items():
    actual = set(datasets[name].columns)
    missing = [col for col in expected if col not in actual]
    extra = [col for col in datasets[name].columns if col not in expected]
    print(f"\n{name.upper()}")
    print("Missing expected columns:", missing)
    print("Extra columns:", extra)

def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return missing count and percentage for each column."""
    return (
        pd.DataFrame({
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_percent": (df.isna().sum().values / len(df) * 100).round(2),
        })
        .sort_values("missing_count", ascending=False)
        .reset_index(drop=True)
    )

for name, df in datasets.items():
    print(f"\nMissing values in {name}:")
    display(missing_value_report(df))



EVENTS
Missing expected columns: []
Extra columns: []

TRANSACTIONS
Missing expected columns: []
Extra columns: []

CUSTOMERS
Missing expected columns: []
Extra columns: []

PRODUCTS
Missing expected columns: []
Extra columns: []

CAMPAIGNS
Missing expected columns: []
Extra columns: []

Missing values in events:


,column,missing_count,missing_percent
0,product_id,200371,10.02
1,device_type,40300,2.02
2,event_id,0,0.00
3,timestamp,0,0.00
4,customer_id,0,0.00
5,session_id,0,0.00
6,event_type,0,0.00
7,traffic_source,0,0.00
8,campaign_id,0,0.00
9,page_category,0,0.00



Missing values in transactions:


,column,missing_count,missing_percent
0,product_id,10449,10.13
1,gross_revenue,10449,10.13
2,transaction_id,0,0.00
3,timestamp,0,0.00
4,customer_id,0,0.00
5,quantity,0,0.00
6,discount_applied,0,0.00
7,campaign_id,0,0.00
8,refund_flag,0,0.00



Missing values in customers:


,column,missing_count,missing_percent
0,customer_id,0,0.0
1,signup_date,0,0.0
2,country,0,0.0
3,age,0,0.0
4,gender,0,0.0
5,loyalty_tier,0,0.0
6,acquisition_channel,0,0.0



Missing values in products:


,column,missing_count,missing_percent
0,product_id,0,0.0
1,category,0,0.0
2,brand,0,0.0
3,base_price,0,0.0
4,launch_date,0,0.0
5,is_premium,0,0.0



Missing values in campaigns:


,column,missing_count,missing_percent
0,campaign_id,0,0.0
1,channel,0,0.0
2,objective,0,0.0
3,start_date,0,0.0
4,end_date,0,0.0
5,target_segment,0,0.0
6,expected_uplift,0,0.0


In [5]:
# -----------------------------
# 3. Date conversion and cleaning
# -----------------------------

events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], errors="coerce")
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
products["launch_date"] = pd.to_datetime(products["launch_date"], errors="coerce")
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"], errors="coerce")
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"], errors="coerce")

date_checks = {
    "events.timestamp": events["timestamp"],
    "transactions.timestamp": transactions["timestamp"],
    "customers.signup_date": customers["signup_date"],
    "products.launch_date": products["launch_date"],
    "campaigns.start_date": campaigns["start_date"],
    "campaigns.end_date": campaigns["end_date"],
}

for label, series in date_checks.items():
    print(f"\n{label}")
    print("Invalid/missing dates:", f"{series.isna().sum():,}")
    print("Minimum:", series.min())
    print("Maximum:", series.max())

events["traffic_source_clean"] = (
    events["traffic_source"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .str.title()
)




events.timestamp
Invalid/missing dates: 0
Minimum: 2021-01-01 00:01:28
Maximum: 2023-12-31 23:57:50

transactions.timestamp
Invalid/missing dates: 0
Minimum: 2021-01-01 00:12:50
Maximum: 2023-12-31 22:37:32

customers.signup_date
Invalid/missing dates: 0
Minimum: 2021-01-01 00:00:00
Maximum: 2023-12-31 00:00:00

products.launch_date
Invalid/missing dates: 0
Minimum: 2021-01-01 00:00:00
Maximum: 2023-12-31 00:00:00

campaigns.start_date
Invalid/missing dates: 0
Minimum: 2021-01-20 00:00:00
Maximum: 2023-11-04 00:00:00

campaigns.end_date
Invalid/missing dates: 0
Minimum: 2021-02-21 00:00:00
Maximum: 2024-01-06 00:00:00


In [6]:

# -----------------------------
# 4. Ad spend availability check
# -----------------------------

spend_keywords = ["spend", "cost", "budget", "cpc", "cpm", "roas", "cac"]

print("\nAd spend-related columns found:")
for name, df in datasets.items():
    spend_columns = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in spend_keywords)
    ]
    print(f"{name}: {spend_columns}")

print(
    "\nConclusion: No real ad spend/cost data is available in the original files. "
    "ROAS and CAC can only be calculated using synthetic spend or future real spend data."
)




Ad spend-related columns found:
events: []
transactions: []
customers: []
products: []
campaigns: []

Conclusion: No real ad spend/cost data is available in the original files. ROAS and CAC can only be calculated using synthetic spend or future real spend data.


In [7]:

# -----------------------------
# 5. Campaign mapping
# -----------------------------

# campaign_id = 0 is not an error. It represents No Campaign / Organic / Direct / Unattributed.
no_campaign_row = pd.DataFrame([{
    "campaign_id": 0,
    "channel": "No Campaign",
    "objective": "Organic/Direct/Unattributed",
    "start_date": pd.NaT,
    "end_date": pd.NaT,
    "target_segment": "Unknown",
    "expected_uplift": 0,
}])

campaigns_extended = pd.concat([campaigns, no_campaign_row], ignore_index=True)

print("Events with campaign_id = 0:", (events["campaign_id"] == 0).sum())
print("Transactions with campaign_id = 0:", (transactions["campaign_id"] == 0).sum())



Events with campaign_id = 0: 1000251
Transactions with campaign_id = 0: 20955


In [8]:

# -----------------------------
# 6. Transaction enrichment and summary metrics
# -----------------------------

transactions_enriched = (
    transactions
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(campaigns_extended, on="campaign_id", how="left")
)

valid_revenue_transactions = transactions_enriched[
    (transactions_enriched["refund_flag"] == 0) &
    (transactions_enriched["gross_revenue"].notna())
].copy()

total_transactions = transactions_enriched["transaction_id"].nunique()
valid_transactions_count = valid_revenue_transactions["transaction_id"].nunique()
unique_buyers = transactions_enriched["customer_id"].nunique()
refund_count = transactions_enriched["refund_flag"].sum()

recorded_revenue_all_transactions = transactions_enriched["gross_revenue"].sum()
revenue_from_non_refunded_transactions = valid_revenue_transactions["gross_revenue"].sum()
refund_flagged_transaction_value = transactions_enriched.loc[
    transactions_enriched["refund_flag"] == 1, "gross_revenue"
].sum()

summary_metrics = pd.DataFrame({
    "metric": [
        "Total Transactions",
        "Valid Revenue Transactions",
        "Unique Buyers",
        "Refund Count",
        "Recorded Revenue - All Transactions",
        "Revenue from Non-Refunded Transactions",
        "Refund-Flagged Transaction Value",
    ],
    "value": [
        total_transactions,
        valid_transactions_count,
        unique_buyers,
        refund_count,
        recorded_revenue_all_transactions,
        revenue_from_non_refunded_transactions,
        refund_flagged_transaction_value,
    ],
})

display(summary_metrics)


,metric,value
0,Total Transactions,103127.00
1,Valid Revenue Transactions,89974.00
2,Unique Buyers,64035.00
3,Refund Count,3029.00
4,Recorded Revenue - All Transactions,8373966.36
5,Revenue from Non-Refunded Transactions,8630269.31
6,Refund-Flagged Transaction Value,-256302.95


In [14]:
# -----------------------------
# 7. Funnel summary
# -----------------------------

funnel_order = ["view", "click", "add_to_cart", "purchase"]

funnel_df = (
    events[events["event_type"].isin(funnel_order)]
    .groupby("event_type")
    .size()
    .reindex(funnel_order)
    .reset_index(name="event_count")
)

funnel_df["stage_order"] = range(1, len(funnel_df) + 1)
funnel_df["previous_stage_count"] = funnel_df["event_count"].shift(1)

funnel_df["conversion_from_previous_percent"] = (
    funnel_df["event_count"] / funnel_df["previous_stage_count"] * 100
).round(2)

funnel_df["dropoff_count"] = (
    funnel_df["previous_stage_count"] - funnel_df["event_count"]
)

# First row has no previous stage
funnel_df["previous_stage_count"] = funnel_df["previous_stage_count"].fillna(0)
funnel_df["conversion_from_previous_percent"] = funnel_df["conversion_from_previous_percent"].fillna(100)
funnel_df["dropoff_count"] = funnel_df["dropoff_count"].fillna(0)

display(funnel_df)
# Save corrected funnel file for Power BI
funnel_df.to_csv(OUTPUT_DIR / "funnel_summary.csv", index=False)

print("Corrected funnel_summary.csv saved successfully.")
display(funnel_df)


,event_type,event_count,stage_order,previous_stage_count,conversion_from_previous_percent,dropoff_count
0,view,1043573,1,0.0,100.00,0.0
1,click,379008,2,1043573.0,36.32,664565.0
2,add_to_cart,284370,3,379008.0,75.03,94638.0
3,purchase,103127,4,284370.0,36.27,181243.0


Corrected funnel_summary.csv saved successfully.


,event_type,event_count,stage_order,previous_stage_count,conversion_from_previous_percent,dropoff_count
0,view,1043573,1,0.0,100.00,0.0
1,click,379008,2,1043573.0,36.32,664565.0
2,add_to_cart,284370,3,379008.0,75.03,94638.0
3,purchase,103127,4,284370.0,36.27,181243.0


In [10]:
# -----------------------------
# 8. Revenue segment tables
# -----------------------------

revenue_by_channel = (
    valid_revenue_transactions
    .groupby("channel", dropna=False)["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"gross_revenue": "net_revenue"})
)

revenue_by_category = (
    valid_revenue_transactions
    .groupby("category", dropna=False)["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"gross_revenue": "net_revenue"})
)

revenue_by_loyalty = (
    valid_revenue_transactions
    .groupby("loyalty_tier", dropna=False)["gross_revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"gross_revenue": "net_revenue"})
)

display(revenue_by_channel)
display(revenue_by_category)
display(revenue_by_loyalty)

,channel,net_revenue
0,No Campaign,1744720.43
1,Affiliate,1652899.35
2,Paid Search,1581814.72
3,Email,1398378.96
4,Display,1248268.04
5,Social,1004187.81


,category,net_revenue
0,Electronics,3554175.61
1,Home,2053954.96
2,Fashion,1338781.78
3,Sports,1000708.47
4,Beauty,381439.05
5,Grocery,301209.44


,loyalty_tier,net_revenue
0,Bronze,4615872.65
1,Silver,2369478.75
2,Gold,1313263.06
3,Platinum,331654.85


In [11]:
# -----------------------------
# 9. Multi-touch attribution
# -----------------------------

events_enriched = events.merge(
    campaigns_extended,
    on="campaign_id",
    how="left"
)

valid_transactions = transactions[
    (transactions["refund_flag"] == 0) &
    (transactions["gross_revenue"].notna())
].copy()

valid_transactions = valid_transactions.rename(columns={
    "timestamp": "transaction_time",
    "campaign_id": "transaction_campaign_id",
})

touchpoint_events = events_enriched[
    events_enriched["event_type"].isin(["view", "click", "add_to_cart"])
].copy()

# Sampling keeps the notebook runnable on a normal laptop.
sample_transactions = valid_transactions.sample(
    n=min(ATTRIBUTION_SAMPLE_SIZE, len(valid_transactions)),
    random_state=RANDOM_SEED
)

touchpoints_sample = sample_transactions.merge(
    touchpoint_events,
    on="customer_id",
    how="left"
)

# Keep events before the transaction and inside the lookback window.
touchpoints_sample = touchpoints_sample[
    (touchpoints_sample["timestamp"] <= touchpoints_sample["transaction_time"]) &
    (
        touchpoints_sample["timestamp"] >=
        (touchpoints_sample["transaction_time"] - pd.Timedelta(days=LOOKBACK_DAYS))
    )
].copy()

# Remove transactions with no valid touchpoints after filtering.
touchpoints_sample = touchpoints_sample[touchpoints_sample["event_id"].notna()].copy()

# First-touch ordering
touchpoints_sample = touchpoints_sample.sort_values(
    ["transaction_id", "timestamp"], ascending=[True, True]
).copy()

touchpoints_sample["touch_order_asc"] = (
    touchpoints_sample.groupby("transaction_id").cumcount() + 1
)

# Last-touch ordering
touchpoints_sample = touchpoints_sample.sort_values(
    ["transaction_id", "timestamp"], ascending=[True, False]
).copy()

touchpoints_sample["touch_order_desc"] = (
    touchpoints_sample.groupby("transaction_id").cumcount() + 1
)

touchpoints_sample["total_touches"] = (
    touchpoints_sample.groupby("transaction_id")["event_id"].transform("count")
)

touchpoints_sample["first_touch_revenue"] = np.where(
    touchpoints_sample["touch_order_asc"] == 1,
    touchpoints_sample["gross_revenue"],
    0
)

touchpoints_sample["last_touch_revenue"] = np.where(
    touchpoints_sample["touch_order_desc"] == 1,
    touchpoints_sample["gross_revenue"],
    0
)

touchpoints_sample["linear_revenue"] = (
    touchpoints_sample["gross_revenue"] / touchpoints_sample["total_touches"]
)

print("Attribution sample rows:", touchpoints_sample.shape[0])
print("Transactions with valid touchpoints:", touchpoints_sample["transaction_id"].nunique())

attribution_by_channel = (
    touchpoints_sample
    .groupby("channel", dropna=False)[[
        "first_touch_revenue",
        "last_touch_revenue",
        "linear_revenue"
    ]]
    .sum()
    .sort_values("linear_revenue", ascending=False)
    .reset_index()
)

attribution_by_campaign = (
    touchpoints_sample
    .groupby(["campaign_id", "channel"], dropna=False)[[
        "first_touch_revenue",
        "last_touch_revenue",
        "linear_revenue"
    ]]
    .sum()
    .sort_values("linear_revenue", ascending=False)
    .reset_index()
)

display(attribution_by_channel)
display(attribution_by_campaign.head(15))


Attribution sample rows: 2287
Transactions with valid touchpoints: 1827


,channel,first_touch_revenue,last_touch_revenue,linear_revenue
0,No Campaign,97154.58,98729.80,98357.303000
1,Email,19363.93,16936.95,18149.544833
2,Affiliate,17232.95,17425.29,17208.536667
3,Paid Search,15965.53,16387.58,16101.316167
4,Display,15174.13,16329.29,15722.986667
5,Social,13874.17,12956.38,13225.602667


,campaign_id,channel,first_touch_revenue,last_touch_revenue,linear_revenue
0,0,No Campaign,97154.58,98729.80,98357.303000
1,21,Email,3566.67,2410.42,2990.623000
2,19,Affiliate,2927.61,2961.09,2890.408333
3,30,Social,3064.00,2218.46,2591.913333
4,4,Display,2679.63,2469.00,2574.315000
5,17,Display,2247.03,2785.64,2546.231667
6,37,Paid Search,2578.53,2164.08,2404.765000
7,43,Paid Search,1620.90,2886.82,2253.860000
8,34,Email,2261.81,1898.73,2059.555000
9,24,Display,1950.08,2091.87,2027.328333


In [12]:
# -----------------------------
# 10. Synthetic ad spend and ROI
# -----------------------------

# These assumptions are synthetic and must not be treated as real business spend.
channel_daily_spend = {
    "Paid Search": 900,
    "Social": 700,
    "Display": 500,
    "Affiliate": 350,
    "Email": 180,
}

channel_ctr_range = {
    "Paid Search": (0.035, 0.070),
    "Social": (0.020, 0.050),
    "Display": (0.008, 0.025),
    "Affiliate": (0.015, 0.040),
    "Email": (0.030, 0.080),
}

objective_multiplier = {
    "Awareness": 0.80,
    "Conversion": 1.25,
    "Retention": 0.75,
    "Cross-sell": 1.00,
    "Acquisition": 1.20,
}

ad_spend_rows = []

for _, row in campaigns.iterrows():  # use original campaigns only; No Campaign does not receive paid spend
    campaign_id = row["campaign_id"]
    channel = row["channel"]
    objective = row["objective"]
    start_date = row["start_date"]
    end_date = row["end_date"]
    expected_uplift = row["expected_uplift"]

    if pd.isna(start_date) or pd.isna(end_date):
        continue

    campaign_dates = pd.date_range(start=start_date, end=end_date, freq="D")
    base_spend = channel_daily_spend.get(channel, 400)
    obj_multiplier = objective_multiplier.get(objective, 1.0)
    uplift_multiplier = 1 + (expected_uplift / 100)

    for date in campaign_dates:
        daily_spend = base_spend * obj_multiplier * uplift_multiplier
        spend = round(np.random.normal(daily_spend, daily_spend * 0.15), 2)
        spend = max(spend, 20)

        cpm_assumption = np.random.uniform(8, 35)
        impressions = int((spend / cpm_assumption) * 1000)

        ctr_low, ctr_high = channel_ctr_range.get(channel, (0.01, 0.04))
        ctr_assumption = np.random.uniform(ctr_low, ctr_high)
        clicks = int(impressions * ctr_assumption)

        ad_spend_rows.append({
            "date": date,
            "campaign_id": campaign_id,
            "channel": channel,
            "objective": objective,
            "impressions": impressions,
            "clicks": clicks,
            "spend": spend,
            "currency": "AED",
        })

ad_spend = pd.DataFrame(ad_spend_rows)

ad_spend["ctr"] = ad_spend["clicks"] / ad_spend["impressions"]
ad_spend["cpc"] = ad_spend["spend"] / ad_spend["clicks"]
ad_spend["cpm"] = (ad_spend["spend"] / ad_spend["impressions"]) * 1000
ad_spend.replace([np.inf, -np.inf], np.nan, inplace=True)

campaign_spend_summary = (
    ad_spend
    .groupby(["campaign_id", "channel", "objective"], dropna=False)
    .agg(
        total_spend=("spend", "sum"),
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
    )
    .reset_index()
)

campaign_spend_summary["ctr"] = (
    campaign_spend_summary["total_clicks"] / campaign_spend_summary["total_impressions"]
)
campaign_spend_summary["cpc"] = (
    campaign_spend_summary["total_spend"] / campaign_spend_summary["total_clicks"]
)
campaign_spend_summary["cpm"] = (
    campaign_spend_summary["total_spend"] / campaign_spend_summary["total_impressions"] * 1000
)
campaign_spend_summary.replace([np.inf, -np.inf], np.nan, inplace=True)

roi_by_campaign = attribution_by_campaign.merge(
    campaign_spend_summary,
    on=["campaign_id", "channel"],
    how="left"
)

campaign_conversions = (
    touchpoints_sample
    .groupby(["campaign_id", "channel"], dropna=False)
    .agg(
        attributed_transactions=("transaction_id", "nunique"),
        attributed_customers=("customer_id", "nunique"),
    )
    .reset_index()
)

roi_by_campaign = roi_by_campaign.merge(
    campaign_conversions,
    on=["campaign_id", "channel"],
    how="left"
)

# ROI metrics are valid only for campaigns with synthetic spend.
roi_by_campaign["first_touch_roas"] = (
    roi_by_campaign["first_touch_revenue"] / roi_by_campaign["total_spend"]
)
roi_by_campaign["last_touch_roas"] = (
    roi_by_campaign["last_touch_revenue"] / roi_by_campaign["total_spend"]
)
roi_by_campaign["linear_roas"] = (
    roi_by_campaign["linear_revenue"] / roi_by_campaign["total_spend"]
)
roi_by_campaign["cost_per_conversion"] = (
    roi_by_campaign["total_spend"] / roi_by_campaign["attributed_transactions"]
)
roi_by_campaign["cac"] = (
    roi_by_campaign["total_spend"] / roi_by_campaign["attributed_customers"]
)

roi_by_campaign.replace([np.inf, -np.inf], np.nan, inplace=True)

# Keep a paid-campaign ROI table for ROI visuals. No Campaign remains in attribution outputs, not ROI outputs.
roi_by_campaign_paid = roi_by_campaign[
    roi_by_campaign["total_spend"].notna() &
    (roi_by_campaign["total_spend"] > 0)
].copy()

roi_by_channel = (
    roi_by_campaign_paid
    .groupby("channel", dropna=False)
    .agg(
        total_spend=("total_spend", "sum"),
        total_impressions=("total_impressions", "sum"),
        total_clicks=("total_clicks", "sum"),
        first_touch_revenue=("first_touch_revenue", "sum"),
        last_touch_revenue=("last_touch_revenue", "sum"),
        linear_revenue=("linear_revenue", "sum"),
        attributed_transactions=("attributed_transactions", "sum"),
        attributed_customers=("attributed_customers", "sum"),
    )
    .reset_index()
)

roi_by_channel["ctr"] = roi_by_channel["total_clicks"] / roi_by_channel["total_impressions"]
roi_by_channel["cpc"] = roi_by_channel["total_spend"] / roi_by_channel["total_clicks"]
roi_by_channel["cpm"] = roi_by_channel["total_spend"] / roi_by_channel["total_impressions"] * 1000

roi_by_channel["first_touch_roas"] = roi_by_channel["first_touch_revenue"] / roi_by_channel["total_spend"]
roi_by_channel["last_touch_roas"] = roi_by_channel["last_touch_revenue"] / roi_by_channel["total_spend"]
roi_by_channel["linear_roas"] = roi_by_channel["linear_revenue"] / roi_by_channel["total_spend"]

roi_by_channel["cost_per_conversion"] = (
    roi_by_channel["total_spend"] / roi_by_channel["attributed_transactions"]
)
roi_by_channel["cac"] = (
    roi_by_channel["total_spend"] / roi_by_channel["attributed_customers"]
)

roi_by_channel.replace([np.inf, -np.inf], np.nan, inplace=True)

display(roi_by_channel.sort_values("linear_roas", ascending=False))

,channel,total_spend,total_impressions,total_clicks,first_touch_revenue,last_touch_revenue,linear_revenue,attributed_transactions,attributed_customers,ctr,cpc,cpm,first_touch_roas,last_touch_roas,linear_roas,cost_per_conversion,cac
2,Email,98785.33,5459538.0,299674.0,19363.93,16936.95,18149.544833,251,251,0.054890,0.329643,18.094082,0.196020,0.171452,0.183727,393.567052,393.567052
0,Affiliate,211606.07,11658978.0,322414.0,17232.95,17425.29,17208.536667,221,221,0.027654,0.656318,18.149624,0.081439,0.082348,0.081323,957.493529,957.493529
1,Display,227777.59,12769675.0,203932.0,15174.13,16329.29,15722.986667,207,205,0.015970,1.116929,17.837383,0.066618,0.071690,0.069028,1100.374831,1111.110195
4,Social,290757.96,16203224.0,562766.0,13874.17,12956.38,13225.602667,183,182,0.034732,0.516659,17.944451,0.047717,0.044561,0.045487,1588.841311,1597.571209
3,Paid Search,480095.83,25984613.0,1357348.0,15965.53,16387.58,16101.316167,218,218,0.052237,0.353701,18.476159,0.033255,0.034134,0.033538,2202.274450,2202.274450


In [13]:
# -----------------------------
# 11. Export dashboard-ready files
# -----------------------------

summary_metrics.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
funnel_df.to_csv(OUTPUT_DIR / "funnel_summary.csv", index=False)
revenue_by_channel.to_csv(OUTPUT_DIR / "revenue_by_channel.csv", index=False)
revenue_by_category.to_csv(OUTPUT_DIR / "revenue_by_category.csv", index=False)
revenue_by_loyalty.to_csv(OUTPUT_DIR / "revenue_by_loyalty.csv", index=False)

attribution_by_channel.to_csv(OUTPUT_DIR / "attribution_by_channel.csv", index=False)
attribution_by_campaign.to_csv(OUTPUT_DIR / "attribution_by_campaign.csv", index=False)

ad_spend.to_csv(OUTPUT_DIR / "synthetic_ad_spend_daily.csv", index=False)
campaign_spend_summary.to_csv(OUTPUT_DIR / "campaign_spend_summary.csv", index=False)
roi_by_campaign_paid.to_csv(OUTPUT_DIR / "roi_by_campaign.csv", index=False)
roi_by_channel.to_csv(OUTPUT_DIR / "roi_by_channel.csv", index=False)

print("All dashboard-ready output files saved successfully.")
print("Output folder:", OUTPUT_DIR.resolve())

All dashboard-ready output files saved successfully.
Output folder: C:\Users\azmia\Marketing-Attribution-ROI-Dashboard\outputs
